# Task 3 - Step 4: Export results

Copies tables to `results/task3/` and figures to `figures/task3/` and builds `results/task3/results.json` (raw values,
full per-epoch histories, configs, freeze manifest, Sketch-access log entries for Task 3).

In [1]:
# ---- Task 3 common header (identical in every Task 3 notebook) ----
# NOTE: this header never loads any Sketch image or label. Only notebook 03 does, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "shared" / "pacs.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared import pacs, pacs_protocol as proto
from shared.config import load_config

T3 = REPO / "task3"
CFG_DIR = T3 / "configs"
EVAL_CFG = __import__("yaml").safe_load((CFG_DIR / "evaluation.yaml").read_text())
SEED = 6304
# Smoke mode (env TASK3_SMOKE=1): 2 epochs x 5 updates per trained run, outputs under _smoke/, and notebook 03 uses
# RANDOM stand-in images and labels, so no Sketch image or label is touched.
SMOKE = os.environ.get("TASK3_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T3 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T3 / "results" / "tables"      # protocol/data-prep files from notebook 00 (same in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T3 / "checkpoints" / SUB            # git-ignored
CACHE = T3 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ERM_DIR = REPO / "task2" / "checkpoints" / "source_only"      # ERM = Task 2 Source-only, loaded, never retrained
TRAIN_RUNS = ["dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
MAIN_RUNS = ["erm", "dan_dg", "sam"]
STUDY_RUNS = ["dan_dg_lambda0.1", "dan_dg", "dan_dg_lambda10"]    # controlled study: lambda_DG in {0.1, 1, 10}
ALL_RUNS = ["erm", "dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
RUN_NAME = {c: load_config(CFG_DIR, c)["run_name"] for c in ALL_RUNS}
LABEL = {"erm": "ERM", "dan_dg": "DAN-DG (λ=1)", "sam": "SAM (ρ=0.05)",
         "dan_dg_lambda0.1": "DAN-DG (λ=0.1)", "dan_dg_lambda10": "DAN-DG (λ=10)"}


def run_dir(n):
    return ERM_DIR if n == "erm" else CKPT / RUN_NAME[n]


def load_trained(n):
    """Model with the selected checkpoint of run ``n`` (ERM: the Task 2 Source-only checkpoint)."""
    from shared.models import build_model
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(7, SEED)
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
import math, shutil
R_OUT = (RES / "export") if SMOKE else (REPO / "results" / "task3")
F_OUT = (RES / "export" / "figures") if SMOKE else (REPO / "figures" / "task3")
R_OUT.mkdir(parents=True, exist_ok=True); F_OUT.mkdir(parents=True, exist_ok=True)
copied = []
for p in sorted(set(TAB.glob("task3_*")) | set(DATA_TAB.glob("task3_sharpness_batch.json")) | set(DATA_TAB.glob("task3_erm_reference.json"))
                | set(DATA_TAB.glob("task3_no_sketch_access_check.json"))):
    shutil.copy2(p, R_OUT / p.name); copied.append(p.name)
for p in sorted(FIG.glob("task3_*")):
    shutil.copy2(p, F_OUT / p.name); copied.append("figures/" + p.name)
print(len(copied), "files copied")

35 files copied


In [3]:
def clean(o):
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, list):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    return o

def csv(name):
    return pd.read_csv(TAB / name)

comp = csv("task3_method_comparison.csv").set_index("config")
log_path = REPO / "data" / "pacs" / "target_label_access_log.jsonl"
access = [json.loads(l) for l in log_path.read_text().splitlines()] if log_path.exists() else []
res = {"task": "task3_domain_generalization", "seed": SEED,
       "protocol": "identical to Task 2 (task3/configs/base.yaml == task2/configs/base.yaml)",
       "erm_reference": json.loads((DATA_TAB / "task3_erm_reference.json").read_text()),
       "no_sketch_access_check": json.loads((DATA_TAB / "task3_no_sketch_access_check.json").read_text()),
       "sharpness_batch": json.loads((DATA_TAB / "task3_sharpness_batch.json").read_text()),
       "freeze_manifest": json.loads((CACHE / "freeze_manifest.json").read_text()),
       "sketch_access_log_task3": [a for a in access if "task3" in a.get("manifest", "")],
       "erm_consistency_with_task2": json.loads((TAB / "task3_erm_consistency.json").read_text()),
       "runs": {}, "controlled_study": csv("task3_controlled_study.csv").to_dict("records"),
       "transfer_cases": csv("task3_transfer_cases.csv").to_dict("records"), "prediction_flips": csv("task3_prediction_flips.csv").to_dict("records"),
       "_notes": {"separability": "3-way multinomial logistic regression (C=1) on balanced source-val features, 70/30, seed 6304; chance 1/3",
                  "sharpness": "L(theta+eps)-L(theta), eps = 0.05 * grad/||grad||, fixed batch of 32 source-val images per domain, eval mode"}}
if (TAB / "task3_task2_comparison.csv").exists():
    res["task2_comparison"] = csv("task3_task2_comparison.csv").to_dict("records")
cls_t, sep, shp = csv("task3_class_analysis.csv"), csv("task3_domain_separability.csv").set_index("config"), csv("task3_sharpness.csv").set_index("config")
for n in ALL_RUNS:
    d = run_dir(n)
    res["runs"][n] = {"label": LABEL[n], "config": load_config(CFG_DIR, n), "training_summary": json.loads((d / "summary.json").read_text()),
                      "history": json.loads((d / "history.json").read_text()), "comparison": comp.loc[n].to_dict(),
                      "source_domain_separability": sep.loc[n].to_dict(), "sharpness": shp.loc[n].to_dict(),
                      "per_class_sketch": cls_t[cls_t.config == n].set_index("class")[["sketch_acc", "change_vs_erm", "n_sketch"]].to_dict("index")}
res["files"] = sorted(copied)
(R_OUT / "results.json").write_text(json.dumps(clean(res), indent=1))
print("wrote", R_OUT / "results.json")
print(comp.loc[MAIN_RUNS, ["mean_acc", "worst_acc", "sketch_acc", "sketch_macro_f1", "sketch_acc_change_vs_erm", "source_domain_separability", "sharpness_delta"]].round(4))

wrote C:\Users\afifh\Desktop\ATML\PA1\results\task3\results.json
        mean_acc  worst_acc  sketch_acc  sketch_macro_f1  \
config                                                     
erm       0.9342     0.9073      0.6747           0.6718   
dan_dg    0.2166     0.1727      0.0407           0.0112   
sam       0.9575     0.9390      0.6007           0.6264   

        sketch_acc_change_vs_erm  source_domain_separability  sharpness_delta  
config                                                                         
erm                       0.0000                      0.8870           0.2839  
dan_dg                   -0.6340                      0.3322           0.0097  
sam                      -0.0741                      0.8904           0.0812  
